## DANH SÁCH CÁC VẤN ĐỀ PHÁT HIỆN VÀ QUYẾT ĐỊNH XỬ LÝ (DECISION LOG)

### 1. Vấn đề 1: Sai kiểu dữ liệu hệ thống (Dtype Mismatch)
* **Quan sát**: Các cột đáng nhẽ phải là số (`Quantity`, `Price Per Unit`, `Total Spent`) và ngày tháng (`Transaction Date`) đều đang ở định dạng chữ (`object`).
* **Nguyên nhân**: Do quá trình nhập liệu bị lẫn lộn các chuỗi văn bản lỗi như 'ERROR', 'UNKNOWN' và khoảng trắng vào các ô số.
* **Quyết định xử lý**: Đồng bộ hóa toàn bộ các chuỗi ký tự lỗi này về định dạng `NaN` mặc định của hệ thống khi đọc file bằng tham số `na_values`, sau đó sử dụng hàm `pd.to_numeric()` và `pd.to_datetime(format='mixed')` để ép về đúng kiểu dữ liệu tính toán.

### 2. Vấn đề 2: Tạp chất và Khuyết thiếu trong cột định lượng (`Quantity`, `Price Per Unit`)
* **Quan sát**: Xuất hiện nhiều ô trống `NaN` và chữ lỗi sau khi ép kiểu.
* **Quyết định xử lý**: Không sử dụng giá trị trung bình (`mean`) tổng thể vì doanh số cà phê phụ thuộc lớn vào từng mặt hàng (ví dụ: Đơn giá của `Cake` khác biệt hoàn toàn với `Cookie`). Tiến hành nhóm dữ liệu theo từng mặt hàng (`groupby('Item')`) và bồi đắp các ô trống bằng giá trị **Trung vị (Median)** của chính nhóm mặt hàng đó để bảo toàn logic kinh tế.

### 3. Vấn đề 3: Lỗi logic toán học ở cột Tổng tiền (`Total Spent`)
* **Quan sát**: Nhiều dòng có số lượng và đơn giá hợp lệ nhưng cột `Total Spent` lại bị để trống hoặc hiển thị chữ `ERROR`.
* **Quyết định xử lý**: Thay vì điền khuyết bằng một con số median cào bằng, ta áp dụng công thức tính toán tường minh: `Total Spent = Quantity * Price Per Unit`. Phương án này giúp khôi phục dữ liệu chính xác 100% theo đúng hóa đơn thực tế của khách hàng.

### 4. Vấn đề 4: Khuyết thiếu thông tin danh mục (`Payment Method`, `Location`)
* **Quan sát**: Cột phương thức thanh toán trống 25.79% và hình thức mua hàng trống 32.65%. Đây là dạng khuyết thiếu hoàn toàn hợp lệ (ví dụ: khách hàng từ chối cung cấp thông tin hoặc hệ thống không bắt buộc điền).
* **Quyết định xử lý**: Điền bồi các ô trống bằng các nhãn hằng số có ý nghĩa nghiệp vụ tương ứng là `'Not Specified'` (Không chỉ định) và `'Unknown'` (Không rõ) nhằm giữ lại nguyên vẹn các hàng phục vụ cho phân tích doanh thu, tránh việc dùng lệnh `.dropna()` làm mất đi 1/3 dữ liệu của hệ thống.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()


def resolve_input_path():
    """Tìm file nguồn theo các vị trí hợp lệ để notebook chạy được ở nhiều máy."""
    candidates = [
        NOTEBOOK_DIR / "dirty_cafe_sales.csv",
        NOTEBOOK_DIR.parent / "lop" / "dirty_cafe_sales.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Không tìm thấy 'dirty_cafe_sales.csv'. Đã kiểm tra: "
        + ", ".join(str(path) for path in candidates)
    )

# ==============================================================================
# 1. ĐỊNH NGHĨA HÀM PIPELINE LÀM SẠCH DỮ LIỆU (TÁI SỬ DỤNG ĐƯỢC)
# ==============================================================================
def clean_cafe_data(file_path):
    """
    Pipeline làm sạch tập dữ liệu dirty_cafe_sales.csv.
    Chuẩn hóa dữ liệu lỗi, chuyển đổi dtype tối ưu, xử lý missing và outliers.
    """
    # Đồng bộ hóa các chuỗi rác thành NaN mặc định của hệ thống
    trash_values = ['ERROR', 'UNKNOWN', 'Unknown', 'error', 'unknown', 'nan', 'NaN']
    df_raw = pd.read_csv(file_path, na_values=trash_values)
    
    # Tạo bản sao tránh sửa in-place trên dữ liệu gốc
    df = df_raw.copy()
    
    # --- BƯỚC 1: XỬ LÝ KIỂU DỮ LIỆU (DTYPE CONVERSION) ---
    df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors='coerce', format='mixed')
    df["Quantity"] = pd.to_numeric(df["Quantity"], errors='coerce')
    df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors='coerce')
    df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors='coerce')
    
    # Chuẩn hóa văn bản thô trước khi đưa vào gán nhóm danh mục
    df["Item"] = df["Item"].astype(str).str.strip().str.title()
    df["Payment Method"] = df["Payment Method"].astype(str).str.strip().str.title()
    df["Location"] = df["Location"].astype(str).str.strip().str.title()
    
    # --- BƯỚC 2: XỬ LÝ TRÙNG LẶP (DUPLICATES) ---
    df = df.drop_duplicates(subset=["Transaction ID"], keep='first').reset_index(drop=True)
    
    # --- BƯỚC 3: XỬ LÝ GIÁ TRỊ KHUYẾT THIẾU VÀ TẠP CHẤT (IMPUTATION) ---
    # 3.1 Cứu dữ liệu Số lượng và Đơn giá dựa trên Trung vị (Median) từng mặt hàng
    df["Quantity"] = df.groupby("Item")["Quantity"].transform(lambda x: x.fillna(x.median()))
    df["Price Per Unit"] = df.groupby("Item")["Price Per Unit"].transform(lambda x: x.fillna(x.median()))
    
    # Fallback toàn cục nếu một nhóm mặt hàng hiếm vẫn không tính được median
    df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())
    df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Price Per Unit"].median())
    
    # 3.2 Khôi phục dựa trên công thức logic hóa đơn thực tế thay vì điền cào bằng
    df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])
    
    # 3.3 Kết hợp ffill() và bfill(); nếu vẫn trống thì dùng mode của toàn cột
    df["Transaction Date"] = df["Transaction Date"].ffill().bfill()
    if df["Transaction Date"].isnull().any():
        fallback_date = df["Transaction Date"].mode(dropna=True)
        if not fallback_date.empty:
            df["Transaction Date"] = df["Transaction Date"].fillna(fallback_date.iloc[0])
    
    # 3.4 Thay thế nhãn rác chuỗi sau biến đổi thành định dạng chuẩn phân loại có nghĩa
    df["Item"] = df["Item"].replace("Nan", "Unspecified")
    df["Payment Method"] = df["Payment Method"].replace("Nan", "Not Specified")
    df["Location"] = df["Location"].replace("Nan", "Unknown")
    
    # Chốt chặn cuối cùng để đảm bảo không còn giá trị khuyết thiếu nào trước unit test
    df["Item"] = df["Item"].fillna("Unspecified")
    df["Payment Method"] = df["Payment Method"].fillna("Not Specified")
    df["Location"] = df["Location"].fillna("Unknown")
    df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])
    
    # --- BƯỚC 4: XỬ LÝ GIÁ TRỊ NGOẠI LAI (OUTLIERS) BẰNG WINSORIZATION ---
    # Sử dụng phương pháp kẹp dữ liệu (Clip) tại phân vị 1st và 99th để xử lý nhiễu đột biến nếu có
    q_low = df["Total Spent"].quantile(0.01)
    q_high = df["Total Spent"].quantile(0.99)
    df["Total Spent"] = df["Total Spent"].clip(lower=q_low, upper=q_high)
    
    # --- BƯỚC 5: TỐI ƯU HÓA BỘ NHỚ THEO CHECKLIST CỦA CÔ (TRANG 3) ---
    df["Item"] = df["Item"].astype('category')
    df["Payment Method"] = df["Payment Method"].astype('category')
    df["Location"] = df["Location"].astype('category')
    
    return df_raw, df


# ==============================================================================
# 2. THỰC THI PIPELINE & ĐO LƯỜNG ĐỘ TIẾT KIỆM BỘ NHỚ RAM
# ==============================================================================
FILE_PATH = resolve_input_path()
df_original, df_clean = clean_cafe_data(FILE_PATH)

# Đo lượng bộ nhớ tiêu thụ của các cột danh mục theo yêu cầu trong Handout
cat_cols = ['Item', 'Payment Method', 'Location']
mem_before = df_original[cat_cols].memory_usage(deep=True).sum()
mem_after = df_clean[cat_cols].memory_usage(deep=True).sum()
saved_pct = ((mem_before - mem_after) / mem_before) * 100


# ==============================================================================
# 3. IN BÁO CÁO CHẤT LƯỢNG ĐỐI CHIẾU TRƯỚC VÀ SAU XỬ LÝ
# ==============================================================================
null_before = df_original.isnull().sum()
for col in df_original.columns:
    null_before[col] += df_original[col].astype(str).str.upper().str.strip().isin(['ERROR', 'UNKNOWN']).sum()

null_after = df_clean.isnull().sum()

summary_df = pd.DataFrame({
    'Cột dữ liệu': df_clean.columns,
    'Kiểu dữ liệu (Trước)': ['object' for _ in df_clean.columns],
    'Kiểu dữ liệu (Sau)': df_clean.dtypes.values,
    'Tổng lượng rác + Null (Trước)': [null_before[col] for col in df_clean.columns],
    'Số lượng Null (Sau)': null_after.values
})

print("==============================================================================")
print("             BẢNG ĐỐI CHIẾU CHẤT LƯỢNG DỮ LIỆU CHI TIẾT THEO CỘT              ")
print("==============================================================================")
print(summary_df.to_string(index=False))
print(f"\n--> [MINH CHỨNG RAM]: Tiết kiệm được {saved_pct:.2f}% bộ nhớ RAM cho các cột danh mục!")
print("==============================================================================")


# ==============================================================================
# 4. UNIT TEST PIPELINE (TIÊU CHÍ BẮT BUỘC ĐỂ LẤY ĐIỂM BONUS - TRANG 7 & 10)
# ==============================================================================
def test_pipeline(df_output):
    """Hàm tự động kiểm tra tính toàn vẹn dữ liệu sau khi chạy qua pipeline."""
    # 1. Khẳng định không còn bất kỳ giá trị khuyết thiếu (Null) nào trong hệ thống
    assert df_output.isnull().sum().sum() == 0, "Unit Test Thất bại: Vẫn còn ô khuyết thiếu!"
    
    # 2. Khẳng định kiểu dữ liệu đã được biến đổi chính xác sang số thực và thời gian
    assert df_output["Quantity"].dtype == 'float64', "Unit Test Thất bại: Cột Quantity sai dtype!"
    assert df_output["Total Spent"].dtype == 'float64', "Unit Test Thất bại: Cột Total Spent sai dtype!"
    assert isinstance(df_output["Transaction Date"].dtype, pd.core.dtypes.dtypes.DatetimeTZDtype) or df_output["Transaction Date"].dtype == 'datetime64[ns]', "Unit Test Thất bại: Cột Date sai định dạng!"
    
    print("\n[UNIT TEST SUCCESS]: Mọi kiểm tra khẳng định (assert) đều vượt qua thành công! Dữ liệu đạt chuẩn 100%.")

# Chạy kiểm thử tự động
test_pipeline(df_clean)

# Xuất file kết quả sạch
df_clean.to_csv(NOTEBOOK_DIR / "cafe_sales_cleaned.csv", index=False)
print("\n-> Hệ thống đã xuất file dữ liệu sạch thành công: 'cafe_sales_cleaned.csv'!")

             BẢNG ĐỐI CHIẾU CHẤT LƯỢNG DỮ LIỆU CHI TIẾT THEO CỘT              
     Cột dữ liệu Kiểu dữ liệu (Trước) Kiểu dữ liệu (Sau)  Tổng lượng rác + Null (Trước)  Số lượng Null (Sau)
  Transaction ID               object                str                              0                    0
            Item               object           category                            969                  969
        Quantity               object            float64                            479                  969
  Price Per Unit               object            float64                            533                  969
     Total Spent               object            float64                            502                   50
  Payment Method               object           category                           3178                 3178
        Location               object           category                           3961                 3961
Transaction Date               object     datetim

AssertionError: Unit Test Thất bại: Vẫn còn ô khuyết thiếu!